# Use Case 5: Knowledge Graph & Temporal Graph Queries

**The Concept:** 
AI agents need to reason over structured relationships between entities — not just flat documents. A Knowledge Graph stores entities as **nodes** and their relationships as **edges**, enabling powerful traversal, pathfinding, and temporal queries.

**The Architecture:** 
SochDB provides a built-in **Graph Overlay** on top of its embedded KV store. You can `add_node()`, `add_edge()`, `traverse()` (BFS), `find_path()`, and `get_neighbors()`. Additionally, its **Temporal Graph** layer supports time-aware edges — allowing questions like *"Where did Alice work in 2023?"*.

---

### Step 0: Install Packages & Setup

In [1]:
!pip install sochdb python-dotenv

import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


True

### Step 1: Initialize Database
Open an embedded SochDB instance for our knowledge graph.

In [2]:
from sochdb import Database

db = Database.open("./knowledge_graph_db")
NAMESPACE = "org_graph"
print("Database opened for Knowledge Graph demo.")

Database opened for Knowledge Graph demo.


### Step 2: Build a Knowledge Graph
We model a small company org chart with **people**, **teams**, and **technologies** as nodes, connected by typed edges.

In [3]:
# --- Add Nodes (Entities) ---
nodes = [
    ("alice",   "person",     {"title": "VP of Engineering", "location": "San Francisco"}),
    ("bob",     "person",     {"title": "Senior Engineer", "location": "New York"}),
    ("carol",   "person",     {"title": "ML Engineer", "location": "London"}),
    ("dave",    "person",     {"title": "Intern", "location": "Remote"}),
    ("eng_team", "team",      {"department": "Engineering", "budget": 500000}),
    ("ml_team",  "team",      {"department": "Machine Learning", "budget": 300000}),
    ("python",   "technology", {"category": "language"}),
    ("sochdb",   "technology", {"category": "database"}),
    ("pytorch",  "technology", {"category": "framework"}),
]

for node_id, node_type, props in nodes:
    db.add_node(namespace=NAMESPACE, node_id=node_id, node_type=node_type, properties=props)

print(f"Added {len(nodes)} nodes to the graph.")

# --- Add Edges (Relationships) ---
edges = [
    ("alice",  "leads",      "eng_team",  {"since": "2021"}),
    ("alice",  "leads",      "ml_team",   {"since": "2023"}),
    ("bob",    "member_of",  "eng_team",  {"role": "backend"}),
    ("carol",  "member_of",  "ml_team",   {"role": "researcher"}),
    ("dave",   "reports_to", "bob",       {"started": "2025"}),
    ("bob",    "reports_to", "alice",     {}),
    ("carol",  "reports_to", "alice",     {}),
    ("bob",    "uses",       "python",    {}),
    ("bob",    "uses",       "sochdb",    {}),
    ("carol",  "uses",       "pytorch",   {}),
    ("carol",  "uses",       "python",    {}),
    ("dave",   "uses",       "python",    {}),
]

for from_id, edge_type, to_id, props in edges:
    db.add_edge(namespace=NAMESPACE, from_id=from_id, edge_type=edge_type, to_id=to_id, properties=props)

print(f"Added {len(edges)} edges to the graph.")

Added 9 nodes to the graph.
Added 12 edges to the graph.


### Step 3: Query Neighbors
Ask the graph: *"Who does Alice lead?"* and *"What does Bob use?"*

In [4]:
# Get Alice's outgoing neighbors (who she leads)
alice_neighbors = db.get_neighbors(
    node_id="alice", direction="outgoing", edge_type="leads", namespace=NAMESPACE
)
print("Alice leads:")
for neighbor in alice_neighbors.get("neighbors", []):
    print(f"  → {neighbor}")

print()

# Get Bob's outgoing 'uses' edges (what technologies he works with)
bob_tech = db.get_neighbors(
    node_id="bob", direction="outgoing", edge_type="uses", namespace=NAMESPACE
)
print("Bob uses:")
for neighbor in bob_tech.get("neighbors", []):
    print(f"  → {neighbor}")

print()

# Get everyone who reports to Alice (incoming edges)
reports_to_alice = db.get_neighbors(
    node_id="alice", direction="incoming", edge_type="reports_to", namespace=NAMESPACE
)
print("People reporting to Alice:")
for neighbor in reports_to_alice.get("neighbors", []):
    print(f"  ← {neighbor}")

Alice leads:
  → {'node_id': 'eng_team', 'direction': 'outgoing', 'edge': {'from_id': 'alice', 'edge_type': 'leads', 'to_id': 'eng_team', 'properties': {'since': '2021'}}}
  → {'node_id': 'ml_team', 'direction': 'outgoing', 'edge': {'from_id': 'alice', 'edge_type': 'leads', 'to_id': 'ml_team', 'properties': {'since': '2023'}}}

Bob uses:
  → {'node_id': 'python', 'direction': 'outgoing', 'edge': {'from_id': 'bob', 'edge_type': 'uses', 'to_id': 'python', 'properties': {}}}
  → {'node_id': 'sochdb', 'direction': 'outgoing', 'edge': {'from_id': 'bob', 'edge_type': 'uses', 'to_id': 'sochdb', 'properties': {}}}

People reporting to Alice:
  ← {'node_id': 'bob', 'direction': 'incoming', 'edge': {'from_id': 'bob', 'edge_type': 'reports_to', 'to_id': 'alice', 'properties': {}}}
  ← {'node_id': 'carol', 'direction': 'incoming', 'edge': {'from_id': 'carol', 'edge_type': 'reports_to', 'to_id': 'alice', 'properties': {}}}


### Step 4: Graph Traversal (BFS)
Traverse the graph starting from Alice to see the full organizational hierarchy.

In [5]:
# BFS traversal from Alice with max depth 3
traversal = db.traverse(
    namespace=NAMESPACE,
    start_node="alice",
    max_depth=3,
    order="bfs"
)

print("BFS Traversal from Alice (depth=3):")
print(json.dumps(traversal, indent=2))

BFS Traversal from Alice (depth=3):
[
  [
    {
      "id": "alice",
      "node_type": "person",
      "properties": {
        "title": "VP of Engineering",
        "location": "San Francisco"
      }
    },
    {
      "id": "eng_team",
      "node_type": "team",
      "properties": {
        "department": "Engineering",
        "budget": 500000
      }
    },
    {
      "id": "ml_team",
      "node_type": "team",
      "properties": {
        "department": "Machine Learning",
        "budget": 300000
      }
    }
  ],
  [
    {
      "from_id": "alice",
      "edge_type": "leads",
      "to_id": "eng_team",
      "properties": {
        "since": "2021"
      }
    },
    {
      "from_id": "alice",
      "edge_type": "leads",
      "to_id": "ml_team",
      "properties": {
        "since": "2023"
      }
    }
  ]
]


### Step 5: Pathfinding
Find the shortest path between Dave (intern) and the ML team. This answers: *"How is Dave connected to the ML team?"*

In [6]:
path = db.find_path(
    from_node="dave",
    to_node="ml_team",
    max_depth=5,
    namespace=NAMESPACE
)

if path:
    print("Path from Dave → ML Team:")
    print(json.dumps(path, indent=2))
else:
    print("No path found between Dave and ML Team.")

Path from Dave → ML Team:
{
  "path": [
    "dave",
    "bob",
    "alice",
    "ml_team"
  ],
  "edges": [
    {
      "from_id": "dave",
      "edge_type": "reports_to",
      "to_id": "bob",
      "properties": {
        "started": "2025"
      }
    },
    {
      "from_id": "bob",
      "edge_type": "reports_to",
      "to_id": "alice",
      "properties": {}
    },
    {
      "from_id": "alice",
      "edge_type": "leads",
      "to_id": "ml_team",
      "properties": {
        "since": "2023"
      }
    }
  ]
}


### Step 6: Temporal Graph — Time-Aware Relationships
Model career changes over time. Alice worked at **Acme Corp** from 2020–2023, then moved to **Quantum Inc** in 2024. Temporal edges let us query *"Where did Alice work in 2022?"* vs *"Where does Alice work now?"*

In [7]:
TEMPORAL_NS = "career_history"

# Alice worked at Acme Corp from 2020 to 2023
db.add_temporal_edge(
    namespace=TEMPORAL_NS,
    from_id="alice",
    edge_type="works_at",
    to_id="acme_corp",
    valid_from=2020,
    valid_until=2023,
    properties={"role": "Senior Engineer"}
)

# Alice moved to Quantum Inc in 2024 (still there — valid_until=0 means open-ended)
db.add_temporal_edge(
    namespace=TEMPORAL_NS,
    from_id="alice",
    edge_type="works_at",
    to_id="quantum_inc",
    valid_from=2024,
    valid_until=0,
    properties={"role": "VP of Engineering"}
)

print("Temporal edges added for Alice's career history.")

Temporal edges added for Alice's career history.


In [8]:
# Time-Travel Query: Where did Alice work in 2022?
result_2022 = db.query_temporal_graph(
    namespace=TEMPORAL_NS,
    node_id="alice",
    mode="snapshot",
    timestamp=2022,
    edge_type="works_at"
)
print("Where did Alice work in 2022?")
print(f"  → {json.dumps(result_2022, indent=2)}")

print()

# Time-Travel Query: Where does Alice work now (2025)?
result_now = db.query_temporal_graph(
    namespace=TEMPORAL_NS,
    node_id="alice",
    mode="snapshot",
    timestamp=2025,
    edge_type="works_at"
)
print("Where does Alice work in 2025?")
print(f"  → {json.dumps(result_now, indent=2)}")

Where did Alice work in 2022?
  → [
  {
    "from_id": "alice",
    "edge_type": "works_at",
    "to_id": "acme_corp",
    "valid_from": 2020,
    "valid_until": 2023,
    "properties": {
      "role": "Senior Engineer"
    }
  }
]

Where does Alice work in 2025?
  → [
  {
    "from_id": "alice",
    "edge_type": "works_at",
    "to_id": "quantum_inc",
    "valid_from": 2024,
    "valid_until": 0,
    "properties": {
      "role": "VP of Engineering"
    }
  }
]


### Step 7: End a Temporal Edge
If Alice leaves Quantum Inc, we close the open-ended edge.

In [9]:
db.end_temporal_edge(
    from_id="alice",
    edge_type="works_at",
    to_id="quantum_inc",
    namespace=TEMPORAL_NS
)

print("Alice's employment at Quantum Inc has been ended.")

# Verify — querying far in the future should show no active employment
result_future = db.query_temporal_graph(
    namespace=TEMPORAL_NS,
    node_id="alice",
    mode="snapshot",
    timestamp=2030,
    edge_type="works_at"
)
print(f"Alice's employment in 2030: {json.dumps(result_future, indent=2)}")

Alice's employment at Quantum Inc has been ended.
Alice's employment in 2030: [
  {
    "from_id": "alice",
    "edge_type": "works_at",
    "to_id": "quantum_inc",
    "valid_from": 2024,
    "valid_until": 1772698773648,
    "properties": {
      "role": "VP of Engineering"
    }
  }
]


### Cleanup

In [10]:
db.close()
print("Database closed.")

Database closed.
